In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from datetime import date
import collections
import datetime
import os
import xarray as xr
from cycler import cycler
import matplotlib.patches as mpatches

Loading in small-scale data:

In [ ]:
#All of the climate models used
modelnames = ['BCC-CSM2-MR', 'CESM2', 'CESM2-WACCM', 'EC-Earth3', 'EC-Earth3-Veg', 'FGOALS-f3-L', 'GFDL-ESM4', 
                  'INM-CM4-8', 'INM-CM5-0', 'MPI-ESM1-2-HR', 'MRI-ESM2-0', 'NorESM2-MM']

scenarios = ['ssp126','ssp245','ssp370','ssp585']

gmodels = ['GloGEM', 'OGGM', 'PyGEM']

cities = ['La Paz', 'Chitina', 'Klutina', 'Geneva', 'Grenoble']

fpath = '/Users/finnwimberly/Desktop/Subbasin Examination/Processed Data/'

In [ ]:
# Initialize annual_RF and percent_change in one line each
annual_RF = {gmodel: {city: {SSP: {GCM: None for GCM in modelnames} for SSP in scenarios} for city in cities} for gmodel in gmodels}
percent_change = {gmodel: {city: {SSP: {GCM: None for GCM in modelnames} for SSP in scenarios} for city in cities} for gmodel in gmodels}

# Loop through the scenarios, basins, models, and GCMs
for SSP in scenarios:
    for gmodel in gmodels:
        for GCM in modelnames:
            for gmodel in gmodels:
                for city in cities:
                    fname = f"Subbasin_RF_{GCM}_{SSP}_{gmodel}_{city}.csv"
                    temp_df = pd.read_csv(fpath + fname, index_col = 0)
                    temp_df.index = pd.to_datetime(temp_df.index, format='%Y-%m-%d').year
                    annual_RF[gmodel][city][SSP][GCM] = temp_df['Annual Runoff [km^3]']
                    percent_change[gmodel][city][SSP][GCM] = temp_df['Percent Change [%]']

In [ ]:
GCM_mean_cities = {gmodel: {city_name: {SSP: {} for SSP in scenarios} for city_name in cities} for gmodel in gmodels}
GCM_q1_cities = {gmodel: {city_name: {SSP: {} for SSP in scenarios} for city_name in cities} for gmodel in gmodels}
GCM_q3_cities = {gmodel: {city_name: {SSP: {} for SSP in scenarios} for city_name in cities} for gmodel in gmodels}

# Taking multi-GCM means and quartiles
for g, gmodel in enumerate(gmodels):
    for city_name in cities: 
        for s, SSP in enumerate(scenarios):
            df_dict = annual_RF[gmodel][city_name][SSP]
            df_mean = pd.concat(df_dict.values(), axis=1).mean(axis=1)
            GCM_mean_cities[gmodel][city_name][SSP] = df_mean

            df_q1 = pd.concat(df_dict.values(), axis=1).quantile(q=0.25, axis=1)
            GCM_q1_cities[gmodel][city_name][SSP] = df_q1

            df_q3 = pd.concat(df_dict.values(), axis=1).quantile(q=0.75, axis=1)
            GCM_q3_cities[gmodel][city_name][SSP] = df_q3

In [ ]:
#Percent Change 

mean_percent_change = {gmodel: {city: {SSP: {GCM: None for GCM in modelnames} for SSP in scenarios} for city in cities} for gmodel in gmodels}
# Loop through the scenarios, basins, models, and GCMs
for SSP in scenarios:
    for gmodel in gmodels:
        for city in cities:
            mean_percent_change[gmodel][city][SSP] = ((GCM_mean_cities[gmodel][city][SSP]- GCM_mean_cities[gmodel][city][SSP][0:20].mean()) /  GCM_mean_cities[gmodel][city][SSP][0:20].mean())*100

Loading in major river basin data:

In [ ]:
basins = {'COPPER' : 'RGI 01', 'RHONE' : 'RGI 11', 'AMAZON' : 'RGI 16', 'TITICACA' : 'RGI 16'}

fpath0 = '/Users/finnwimberly/Desktop/Lizz Research/CSV Outputs/Load Separate/'

In [ ]:
# Create new index using pandas date_range function
start_date = datetime.date(2000, 1, 1)
end_date = datetime.date(2100, 12, 1)
new_indices = pd.date_range(start_date, end_date, freq='MS').strftime('%Y-%m').tolist()

In [ ]:
#Loading data, applying datetime indices, and creating annual sum dfs
all_rf_data = {}
all_rf_data_annual = {}
for g, gmodel in enumerate(gmodels):
    all_rf_data[gmodel] = {}
    all_rf_data_annual[gmodel] = {}
    fpath = fpath0 + gmodel + '/'
    for s, SSP in enumerate(scenarios):
        all_rf_data[gmodel][SSP] = {}
        all_rf_data_annual[gmodel][SSP] = {}
        for basin, region in basins.items():
            all_rf_data[gmodel][SSP][basin] = {}
            all_rf_data_annual[gmodel][SSP][basin] = {}
            for m, GCM in enumerate(modelnames):
                fname = f"{region}/{gmodel}/runoff_{GCM}_{SSP}_{basin}.csv"
                temp_df = pd.read_csv(fpath0 + fname, index_col = 0)
                all_rf_data[gmodel][SSP][basin][GCM] = temp_df
                all_rf_data[gmodel][SSP][basin][GCM].index = new_indices
                all_rf_data[gmodel][SSP][basin][GCM].index = pd.to_datetime(new_indices)
                all_rf_data_annual[gmodel][SSP][basin][GCM] = all_rf_data[gmodel][SSP][basin][GCM].resample('A').sum()

In [ ]:
GCM_mean_basins = {gmodel: {basin: {SSP: {} for SSP in scenarios} for basin in basins} for gmodel in gmodels}
GCM_q1_basins = {gmodel: {basin: {SSP: {} for SSP in scenarios} for basin in basins} for gmodel in gmodels}
GCM_q3_basins = {gmodel: {basin: {SSP: {} for SSP in scenarios} for basin in basins} for gmodel in gmodels}

# Taking multi-GCM means and quartiles
for g, gmodel in enumerate(gmodels):
    for basin, region in basins.items():
        for s, SSP in enumerate(scenarios):
            df_dict = all_rf_data_annual[gmodel][SSP][basin]
           
            df_mean = pd.concat(df_dict.values(), axis=1).mean(axis=1)
            GCM_mean_basins[gmodel][basin][SSP] = df_mean

            df_q1 = pd.concat(df_dict.values(), axis=1).quantile(q=0.25, axis=1)
            GCM_q1_basins[gmodel][basin][SSP] = df_q1

            df_q3 = pd.concat(df_dict.values(), axis=1).quantile(q=0.75, axis=1)
            GCM_q3_basins[gmodel][basin][SSP] = df_q3

In [ ]:
#Calculating percent change
percent_change_basin = {gmodel: {SSP: {basin: {} for basin in basins} for SSP in scenarios} for gmodel in gmodels}

for gmodel in gmodels:
    for SSP in scenarios:
        for basin in basins:
            percent_change_basin[gmodel][SSP][basin] = ((GCM_mean[gmodel][SSP][basin]- GCM_mean[gmodel][SSP][basin][0:20].mean()) /  GCM_mean[gmodel][SSP][basin].mean())*100

Making plots:

In [ ]:
#Creating time values and color schemes
yrs = np.arange(2000, 2101)
yrs_dt = pd.to_datetime([str(y) for y in yrs])

colorschemes = {}

colors_glo =  plt.colormaps['Greens']
line_colors_glo = colors_glo(np.linspace(0.2, 0.6, num = 12))
glo_cycler = cycler(color = line_colors_glo)
colorschemes['GloGEM'] = glo_cycler

colors_OG =  plt.colormaps['Blues']
line_colors_OG = colors_OG(np.linspace(0.2, 0.6,num = 12))
OG_cycler = cycler(color = line_colors_OG)
colorschemes['OGGM'] = OG_cycler

colors_py =  plt.colormaps['Purples']
line_colors_py = colors_py(np.linspace(0.2, 0.6,num = 12))
py_cycler = cycler(color = line_colors_py)
colorschemes['PyGEM'] = py_cycler

colors = {'GloGEM': 'darkgreen', 'OGGM': 'royalblue', 'PyGEM': 'purple'}
fill_colors = {'GloGEM': 'green', 'OGGM': 'dodgerblue', 'PyGEM': 'purple'}

In [ ]:
SSP = 'ssp245'
city = 'La Paz'
fig, axs = plt.subplots(3, 2, figsize=(12, 12), sharex=True)

# Top row
for g, gmodel in enumerate(gmodels):
        for m, GCM in enumerate(modelnames):
            axs[0, 0].plot(yrs_dt[0:-1], annual_RF[gmodel][city][SSP][GCM], color=axs[0, 0].set_prop_cycle(colorschemes[gmodel]), alpha=0.25)
            axs[0, 0].plot(yrs_dt[0:-1], GCM_mean_cities[gmodel][city][SSP], color=colors[gmodel], linewidth=0.9)
            axs[0, 0].plot(yrs_dt[0:-1], GCM_q1_cities[gmodel][city][SSP], color=colors[gmodel], linewidth=0.4)
            axs[0, 0].plot(yrs_dt[0:-1], GCM_q3_cities[gmodel][city][SSP], color=colors[gmodel], linewidth=0.4)
            axs[0, 0].fill_between(yrs_dt[0:-1], GCM_q1_cities[gmodel][city][SSP], GCM_q3_cities[gmodel][city][SSP], color=fill_colors[gmodel])
            axs[0, 0].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))
            axs[0, 0].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation=45)
            axs[0, 0].set_title(r'Annual Runoff $[km^3]$', fontsize=10)
            axs[0, 0].set_ylabel('La Paz Watershed', fontsize=10)

            axs[0, 1].plot(yrs_dt[0:-1], mean_percent_change[gmodel][city][SSP].rolling(20, center=True).mean(), color=colors[gmodel], linewidth=0.8)
            axs[0, 1].plot(yrs_dt[0:-1], mean_percent_change[gmodel][city][SSP], color=colors[gmodel], alpha=0.03, linewidth=0.7)
            axs[0, 1].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))
            axs[0, 1].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation=45)
            axs[0, 1].set_title(r'Percent Change $[\%]$', fontsize=10)

# Second row
basins = ['AMAZON', 'TITICACA']
basinstext = ['Amazon', 'Titicaca']
for b, basin in enumerate(basins):
    for g, gmodel in enumerate(gmodels):
        for m, GCM in enumerate(modelnames):
            axs[b+1, 0].plot(yrs_dt[0:-1], all_rf_data_annual[gmodel][SSP][basin][GCM][0:-1], color=axs[b+1, 0].set_prop_cycle(colorschemes[gmodel]), alpha=0.25)
            axs[b+1, 0].plot(yrs_dt[0:-1], GCM_mean_basins[gmodel][basin][SSP][0:-1], color=colors[gmodel], linewidth=0.9)
            axs[b+1, 0].plot(yrs_dt[0:-1], GCM_q1_basins[gmodel][basin][SSP][0:-1], color=colors[gmodel], linewidth=0.4)
            axs[b+1, 0].plot(yrs_dt[0:-1], GCM_q3_basins[gmodel][basin][SSP][0:-1], color=colors[gmodel], linewidth=0.4)
            axs[b+1, 0].fill_between(yrs_dt[0:-1], GCM_q1_basins[gmodel][basin][SSP][0:-1], GCM_q3_basins[gmodel][basin][SSP][0:-1], color=fill_colors[gmodel])
            axs[b+1, 0].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))
            axs[b+1, 0].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation=45)
            axs[b+1, 0].set_ylabel(f"{basinstext[b]} River Basin", fontsize=10)
            if b == 1:
                axs[b+1, 0].set_xlabel('Year')

            axs[b+1, 1].plot(yrs_dt[0:-1], percent_change_basin[gmodel][SSP][basin][0:-1].rolling(20, center=True).mean(), color=colors[gmodel], linewidth=0.8)
            axs[b+1, 1].plot(yrs_dt[0:-1], percent_change_basin[gmodel][SSP][basin][0:-1], color=colors[gmodel], alpha=0.03, linewidth=0.7)
            axs[b+1, 1].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))
            axs[b+1, 1].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation=45)
            if b == 1:
                axs[b+1, 1].set_xlabel('Year')

plt.tight_layout()
name = 'SouthAmerica'
plt.savefig(f"/Users/finnwimberly/Desktop/Subbasin Examination/Comparison Figs/{name}.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
SSP = 'ssp245'
cities = ['Geneva', 'Grenoble']
fig, axs = plt.subplots(3, 2, figsize=(12, 12), sharex=True)

# Top row
for g, gmodel in enumerate(gmodels):
    for c, city in enumerate (cities):
        for m, GCM in enumerate(modelnames):
            axs[c, 0].plot(yrs_dt[0:-1], annual_RF[gmodel][city][SSP][GCM], color=axs[c, 0].set_prop_cycle(colorschemes[gmodel]), alpha=0.25)
            axs[c, 0].plot(yrs_dt[0:-1], GCM_mean_cities[gmodel][city][SSP], color=colors[gmodel], linewidth=0.9)
            axs[c, 0].plot(yrs_dt[0:-1], GCM_q1_cities[gmodel][city][SSP], color=colors[gmodel], linewidth=0.4)
            axs[c, 0].plot(yrs_dt[0:-1], GCM_q3_cities[gmodel][city][SSP], color=colors[gmodel], linewidth=0.4)
            axs[c, 0].fill_between(yrs_dt[0:-1], GCM_q1_cities[gmodel][city][SSP], GCM_q3_cities[gmodel][city][SSP], color=fill_colors[gmodel])
            axs[c, 0].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))
            axs[c, 0].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation=45)
            axs[0, 0].set_title(r'Annual Runoff $[km^3]$', fontsize=10)
            axs[c, 0].set_ylabel(f'{city} Watershed', fontsize=10)

            axs[c, 1].plot(yrs_dt[0:-1], mean_percent_change[gmodel][city][SSP].rolling(20, center=True).mean(), color=colors[gmodel], linewidth=0.8)
            axs[c, 1].plot(yrs_dt[0:-1], mean_percent_change[gmodel][city][SSP], color=colors[gmodel], alpha=0.03, linewidth=0.7)
            axs[c, 1].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))
            axs[c, 1].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation=45)
            axs[0, 1].set_title(r'Percent Change $[\%]$', fontsize=10)

# Second row
basins = ['RHONE']
basinstext = ['Rhone']
for b, basin in enumerate(basins):
    for g, gmodel in enumerate(gmodels):
        for m, GCM in enumerate(modelnames):
            axs[b+2, 0].plot(yrs_dt[0:-1], all_rf_data_annual[gmodel][SSP][basin][GCM][0:-1], color=axs[b+2, 0].set_prop_cycle(colorschemes[gmodel]), alpha=0.25)
            axs[b+2, 0].plot(yrs_dt[0:-1], GCM_mean_basins[gmodel][basin][SSP][0:-1], color=colors[gmodel], linewidth=0.9)
            axs[b+2, 0].plot(yrs_dt[0:-1], GCM_q1_basins[gmodel][basin][SSP][0:-1], color=colors[gmodel], linewidth=0.4)
            axs[b+2, 0].plot(yrs_dt[0:-1], GCM_q3_basins[gmodel][basin][SSP][0:-1], color=colors[gmodel], linewidth=0.4)
            axs[b+2, 0].fill_between(yrs_dt[0:-1], GCM_q1_basins[gmodel][basin][SSP][0:-1], GCM_q3_basins[gmodel][basin][SSP][0:-1], color=fill_colors[gmodel])
            axs[b+2, 0].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))
            axs[b+2, 0].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation=45)
            axs[b+2, 0].set_ylabel(f"{basinstext[b]} River Basin", fontsize=10)
            axs[b+2, 0].set_xlabel('Year')

            axs[b+2, 1].plot(yrs_dt[0:-1], percent_change_basin[gmodel][SSP][basin][0:-1].rolling(20, center=True).mean(), color=colors[gmodel], linewidth=0.8)
            axs[b+2, 1].plot(yrs_dt[0:-1], percent_change_basin[gmodel][SSP][basin][0:-1], color=colors[gmodel], alpha=0.03, linewidth=0.7)
            axs[b+2, 1].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))
            axs[b+2, 1].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation=45)
            axs[b+2, 1].set_xlabel('Year')


plt.tight_layout()
name = 'Europe'
plt.savefig(f"/Users/finnwimberly/Desktop/Subbasin Examination/Comparison Figs/{name}.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
SSP = 'ssp245'
cities = ['Chitina', 'Klutina']
fig, axs = plt.subplots(3, 2, figsize=(12, 12), sharex=True)

# Top row
for g, gmodel in enumerate(gmodels):
    for c, city in enumerate (cities):
        for m, GCM in enumerate(modelnames):
            axs[c, 0].plot(yrs_dt[0:-1], annual_RF[gmodel][city][SSP][GCM], color=axs[c, 0].set_prop_cycle(colorschemes[gmodel]), alpha=0.25)
            axs[c, 0].plot(yrs_dt[0:-1], GCM_mean_cities[gmodel][city][SSP], color=colors[gmodel], linewidth=0.9)
            axs[c, 0].plot(yrs_dt[0:-1], GCM_q1_cities[gmodel][city][SSP], color=colors[gmodel], linewidth=0.4)
            axs[c, 0].plot(yrs_dt[0:-1], GCM_q3_cities[gmodel][city][SSP], color=colors[gmodel], linewidth=0.4)
            axs[c, 0].fill_between(yrs_dt[0:-1], GCM_q1_cities[gmodel][city][SSP], GCM_q3_cities[gmodel][city][SSP], color=fill_colors[gmodel])
            axs[c, 0].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))
            axs[c, 0].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation=45)
            axs[0, 0].set_title(r'Annual Runoff $[km^3]$', fontsize=10)
            axs[c, 0].set_ylabel(f'{city} Watershed', fontsize=10)

            axs[c, 1].plot(yrs_dt[0:-1], mean_percent_change[gmodel][city][SSP].rolling(20, center=True).mean(), color=colors[gmodel], linewidth=0.8)
            axs[c, 1].plot(yrs_dt[0:-1], mean_percent_change[gmodel][city][SSP], color=colors[gmodel], alpha=0.03, linewidth=0.7)
            axs[c, 1].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))
            axs[c, 1].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation=45)
            axs[0, 1].set_title(r'Percent Change $[\%]$', fontsize=10)

# Second row
basins = ['COPPER']
basinstext = ['Copper']
for b, basin in enumerate(basins):
    for g, gmodel in enumerate(gmodels):
        for m, GCM in enumerate(modelnames):
            axs[b+2, 0].plot(yrs_dt[0:-1], all_rf_data_annual[gmodel][SSP][basin][GCM][0:-1], color=axs[b+2, 0].set_prop_cycle(colorschemes[gmodel]), alpha=0.25)
            axs[b+2, 0].plot(yrs_dt[0:-1], GCM_mean_basins[gmodel][basin][SSP][0:-1], color=colors[gmodel], linewidth=0.9)
            axs[b+2, 0].plot(yrs_dt[0:-1], GCM_q1_basins[gmodel][basin][SSP][0:-1], color=colors[gmodel], linewidth=0.4)
            axs[b+2, 0].plot(yrs_dt[0:-1], GCM_q3_basins[gmodel][basin][SSP][0:-1], color=colors[gmodel], linewidth=0.4)
            axs[b+2, 0].fill_between(yrs_dt[0:-1], GCM_q1_basins[gmodel][basin][SSP][0:-1], GCM_q3_basins[gmodel][basin][SSP][0:-1], color=fill_colors[gmodel])
            axs[b+2, 0].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))
            axs[b+2, 0].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation=45)
            axs[b+2, 0].set_ylabel(f"{basinstext[b]} River Basin", fontsize=10)
            axs[b+2, 0].set_xlabel('Year')

            axs[b+2, 1].plot(yrs_dt[0:-1], percent_change_basin[gmodel][SSP][basin][0:-1].rolling(20, center=True).mean(), color=colors[gmodel], linewidth=0.8)
            axs[b+2, 1].plot(yrs_dt[0:-1], percent_change_basin[gmodel][SSP][basin][0:-1], color=colors[gmodel], alpha=0.03, linewidth=0.7)
            axs[b+2, 1].set(xlim=(pd.to_datetime('2000-01-01'), pd.to_datetime('2100-01-01')))
            axs[b+2, 1].set_xticks([pd.to_datetime('2000'), pd.to_datetime('2025'), pd.to_datetime('2050'), pd.to_datetime('2075'), pd.to_datetime('2100')], [2000, 2025, 2050, 2075, 2100], rotation=45)
            axs[b+2, 1].set_xlabel('Year')


plt.tight_layout()
name = 'NorthAmerica'
plt.savefig(f"/Users/finnwimberly/Desktop/Subbasin Examination/Comparison Figs/{name}.png", dpi=300, bbox_inches='tight')
plt.show()

Calculating some summary stats to compare basins and subbasins:

Loading in total basin areas

In [ ]:
## Generic the filepath to the main data folder
fpath0 = '/Users/finnwimberly/Library/CloudStorage/GoogleDrive-fwimberly@middlebury.edu/My Drive/'
fpath1 = 'Lizz Research Stuff/RF Intercomp/CSV Outputs/'

In [ ]:
#Loading in total Basin area data
from scipy.io import loadmat
BasinAreas = loadmat(fpath0 + fpath1 + 'BasinArea.mat')

#Creating indexed df
basin_areas = BasinAreas['BasinArea']
basin_names = BasinAreas['BasinNam']
basin_name_list = [name[1][0] for name in basin_names]

TotalBasinAreas = pd.DataFrame({'Basin Area': basin_areas.squeeze()}, index=basin_name_list)

In [ ]:
subbasin_areas = {'Chitina':21717.5, 'Klutina': 24160.3, 'Geneva': 9491.6, 'Grenoble': 4598.3, 'La Paz': 1149.5}
basin_areas = {'Copper': TotalBasinAreas.loc['COPPER', 'Basin Area'],'Rhone': TotalBasinAreas.loc['RHONE', 'Basin Area'],
    'Amazon': TotalBasinAreas.loc['AMAZON', 'Basin Area'],'Titicaca': TotalBasinAreas.loc['TITICACA', 'Basin Area']}

In [ ]:
# Create lists for basin names, subbasin names, and their corresponding areas
basin_names = list(basin_areas.keys())
basin_area_values = list(basin_areas.values())
subbasin_names = list(subbasin_areas.keys())
subbasin_area_values = list(subbasin_areas.values())

# Create a DataFrame with the necessary columns
df = pd.DataFrame({
    'Basin': basin_names + subbasin_names,
    r'Area $[km^2]$': basin_area_values +subbasin_area_values
})

In [ ]:
# Assuming your DataFrame is named 'df' and you have specified the file path
output_file = '/Users/finnwimberly/Desktop/Subbasin Examination/Areas.csv'

# Save DataFrame to CSV
df.to_csv(output_file, index=False)  # Set index=False if you don't want to save the index as a column